In [0]:
from pyspark.sql.functions import *

catalog = "bike_data"
silver_schema = "silver"
gold_schema = "gold"

print("=" * 80)
print("GOLD TRANSFORMATION: dim_products")
print("=" * 80)

# Section 1: Read Silver tables
print("\nSection 1: Reading Silver tables")
products_df = spark.table(f"{catalog}.{silver_schema}.products")
categories_df = spark.table(f"{catalog}.{silver_schema}.categories")

print(f"  products: {products_df.count():,} rows")
print(f"  categories: {categories_df.count():,} rows")

# Section 2: Build dim_products
print("\nSection 2: Building dim_products")

# Start with products
dim_products = products_df.select(
    col("prd_key"),
    col("prd_id"),
    col("prd_nm"),
    col("prd_cost"),
    col("prd_line"),
    col("prd_start_dt"),
    col("prd_end_dt")
)

print(f"  Starting with products: {dim_products.count():,}")

# Left join with categories to add product category info
print("  Joining with categories...")
cat_clean = categories_df.select(
    col("ID"),
    col("CAT"),
    col("SUBCAT"),
    col("MAINTENANCE")
)

dim_products = dim_products.join(
    cat_clean,
    dim_products.prd_id.cast("string") == cat_clean.ID,
    "left"
).drop("ID")

print(f"  After categories join: {dim_products.count():,}")

# Section 3: Sanity checks
print("\nSection 3: Sanity checks")

print("  Null values:")
null_check = dim_products.select([count(when(col(c).isNull(), c)).alias(c) for c in dim_products.columns])
display(null_check)

print("\n  Data sample:")
display(dim_products.limit(3))

print("\n  Schema:")
dim_products.printSchema()

# Section 4: Write to Gold
print("\nSection 4: Writing to Gold table")
gold_table = f"{catalog}.{gold_schema}.dim_products"
dim_products.write.mode("overwrite").format("delta").saveAsTable(gold_table)

final_count = dim_products.count()
print(f"\nWritten to: {gold_table}")
print(f"Row count: {final_count:,}")